In [ ]:
# ============================================================
# Install and Setup MuJoCo in Google Colab
# ============================================================

# Update package lists and install required system graphics libraries
!apt-get update -qq
!apt-get install -y -qq libglfw3 libglew2.0 libgl1-mesa-glx libosmesa6

# Install MuJoCo and Gymnasium (with MuJoCo support)
!pip install -q mujoco
!pip install -q "gymnasium[mujoco]"

print("✅ MuJoCo and Gymnasium installed successfully!")

# Test the installation
import gymnasium as gym

try:
    env = gym.make("HalfCheetah-v5")
    print("✅ HalfCheetah-v5 environment created successfully!")
    print(f"   Observation Space: {env.observation_space.shape}")
    print(f"   Action Space: {env.action_space.shape}")
    env.close()
except Exception as e:
    print(f"❌ Error creating the environment: {e}")

In [ ]:
# ============================================================
# MCDM-DQN (VIKOR Adaptive) on HalfCheetah-v5
# Final version: Fixed constant Recency and low TD-error weight issue
# ============================================================

import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, mannwhitneyu, spearmanr
import zipfile
import shutil
from google.colab import files

# ========== Config ==========
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ENV_NAME = "HalfCheetah-v5"
NUM_RUNS = 10
NUM_EPISODES = 200
BUFFER_SIZE = 100000
BATCH_SIZE = 64
MIN_REPLAY_SIZE = 1000
GAMMA = 0.99
LR = 3e-4
SYNC_TARGET_STEPS = 200
EPS_START = 1.0
EPS_FINAL = 0.05
EPS_DECAY = 50000

# ===== MCDM Parameters =====
ALPHA = 0.6
BETA_START = 0.4
BETA_FRAMES = 100000
PRIO_EPS = 1e-6
WEIGHT_RECOMP_FREQ = 50
ETA_WEIGHT = 0.05

MIN_SAMPLES_FOR_WEIGHT_UPDATE = 500
MIN_SAMPLES_FOR_VIKOR = 50

NUM_DISCRETE_ACTIONS = 3
DISCRETE_ACTIONS_MAP = {
    0: np.full(6, -1.0),
    1: np.full(6, 0.0),
    2: np.full(6, 1.0)
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ============================================================
# VIKOR Function (Final version: Spearman + Absolute value)
# ============================================================
def vikor_weights(criteria_matrix):
    """
    Compute adaptive VIKOR weights.
    - criteria_matrix: Each column is a criterion, each row is a sample. Values must be normalized [0,1].
    - Output: Weight for each criterion (sum = 1)
    """
    if criteria_matrix.shape[0] < MIN_SAMPLES_FOR_VIKOR:
        return np.ones(criteria_matrix.shape[1], dtype=np.float32) / criteria_matrix.shape[1]

    n_samples, n_criteria = criteria_matrix.shape
    f_star = np.max(criteria_matrix, axis=0)
    f_minus = np.min(criteria_matrix, axis=0)
    diff = f_star - f_minus + 1e-12
    norm = (f_star - criteria_matrix) / diff

    w_init = np.ones(n_criteria, dtype=np.float32) / n_criteria
    S_per_sample = (norm * w_init).sum(axis=1)
    R_per_sample = (norm * w_init).max(axis=1)

    S_star, S_minus = np.min(S_per_sample), np.max(S_per_sample)
    R_star, R_minus = np.min(R_per_sample), np.max(R_per_sample)

    Q_per_sample = 0.5 * (S_per_sample - S_star) / (S_minus - S_star + 1e-12) + \
                   0.5 * (R_per_sample - R_star) / (R_minus - R_star + 1e-12)

    weights = np.zeros(n_criteria, dtype=np.float32)
    for j in range(n_criteria):
        try:
            corr, _ = spearmanr(Q_per_sample, criteria_matrix[:, j])
        except Exception:
            corr = 0.0
        if not np.isfinite(corr):
            corr = 0.0
        weights[j] = max(0.05, abs(corr))

    s = weights.sum()
    if s <= 0 or not np.isfinite(s):
        weights = np.ones(n_criteria, dtype=np.float32) / n_criteria
    else:
        weights = weights / s

    return weights


# ========== Q-Network ==========
class QNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim)
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# Replay Buffer (Final version with true Recency)
# ============================================================
class MCDMPrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6, n_criteria=4):
        self.capacity = int(capacity)
        self.alpha = alpha
        self.buffer = []
        self.pos = 0
        self.priorities = np.zeros((self.capacity,), dtype=np.float32)
        self.criteria_raw = np.zeros((self.capacity, n_criteria), dtype=np.float32)
        # Store insertion time for each experience to compute true Recency
        self.insertion_times = np.zeros((self.capacity,), dtype=np.int64)
        self.current_len = 0
        self.weights_cache = np.array([0.25, 0.25, 0.25, 0.25], dtype=np.float32)
        self.push_count = 0
        self.weights_history = [(0, self.weights_cache.copy())]
        self.reward_min = 0.0
        self.reward_max = 0.0

    def _compute_criteria_raw(self, td_error, reward, novelty, recency):
        """
        Raw criteria (without normalization).
        Note: In VIKOR, "lower is better".
        - TD-error: lower is better.
        - Reward: higher is better -> inverted.
        - Novelty: lower is better.
        - Recency: newer is better -> lower value = newer.
        """
        self.reward_min = min(self.reward_min, reward)
        self.reward_max = max(self.reward_max, reward)
        rng = self.reward_max - self.reward_min + 1e-12
        reward_crit = (self.reward_max - reward) / rng

        return np.array(
            [float(abs(td_error)), float(reward_crit), float(novelty), float(recency)],
            dtype=np.float32
        )

    def _recompute_recency_for_all(self):
        """
        Compute true Recency for all experiences:
        - Newest experience (insertion_time = push_count) -> recency = 0 (best).
        - Oldest experience -> recency near 1 (worst).
        """
        n = self.current_len
        if n == 0:
            return
        ages = (self.push_count - self.insertion_times[:n]).astype(np.float32)
        ages = np.maximum(ages, 0)
        max_age = ages.max() + 1e-12
        recencies = ages / max_age  # 0 = newest, 1 = oldest
        self.criteria_raw[:n, 3] = recencies

    def push(self, state, action, reward, next_state, done, td_error=1.0):
        # Novelty
        if self.current_len < 10:
            novelty = 1.0
        else:
            sample_size = min(50, self.current_len)
            sample_indices = np.random.choice(self.current_len, sample_size, replace=False)
            sample_states = np.array([self.buffer[i][0] for i in sample_indices])
            distances = np.linalg.norm(sample_states - state, axis=1)
            novelty = float(np.min(distances))

        # New experience -> newest -> recency = 0 (best in VIKOR)
        recency = 0.0

        crit_raw = self._compute_criteria_raw(td_error, reward, novelty, recency)

        data = (state.copy() if isinstance(state, np.ndarray) else state,
                action, reward,
                next_state.copy() if isinstance(next_state, np.ndarray) else next_state,
                done)

        if self.current_len < self.capacity:
            self.buffer.append(data)
            self.current_len += 1
        else:
            self.buffer[self.pos] = data
        self.criteria_raw[self.pos] = crit_raw
        self.insertion_times[self.pos] = self.push_count

        # ===== Update Weights =====
        if (self.current_len >= MIN_SAMPLES_FOR_WEIGHT_UPDATE) and \
           (self.push_count % WEIGHT_RECOMP_FREQ == 0):
            try:
                # Recompute Recency for all before VIKOR
                self._recompute_recency_for_all()

                crit_matrix = self.criteria_raw[:self.current_len].copy()
                col_min = crit_matrix.min(axis=0, keepdims=True)
                col_max = crit_matrix.max(axis=0, keepdims=True)
                col_rng = col_max - col_min + 1e-12
                crit_matrix_norm = (crit_matrix - col_min) / col_rng

                new_weights = vikor_weights(crit_matrix_norm)

                # Equation 5: w = (1-η)*w + η*new_weights
                self.weights_cache = (1 - ETA_WEIGHT) * self.weights_cache + ETA_WEIGHT * new_weights
                # Equation 6: Normalization
                self.weights_cache = self.weights_cache / (self.weights_cache.sum() + 1e-12)

                if self.push_count % (WEIGHT_RECOMP_FREQ * 20) == 0:
                    self.weights_history.append((self.push_count, self.weights_cache.copy()))
            except Exception:
                pass

        # Priority = weighted sum of normalized criteria
        crit_norm = crit_raw / (crit_raw.sum() + 1e-12)
        prio = float(np.dot(crit_norm, self.weights_cache))
        if not np.isfinite(prio) or prio <= 0:
            prio = PRIO_EPS
        self.priorities[self.pos] = prio

        self.pos = (self.pos + 1) % self.capacity
        self.push_count += 1

    def sample(self, batch_size, beta=0.4):
        if self.current_len == 0:
            return None

        prios = self.priorities[:self.current_len].copy()
        prios[~np.isfinite(prios)] = PRIO_EPS
        prios = np.clip(prios, PRIO_EPS, None)

        probs = prios ** self.alpha
        s = probs.sum()
        if s <= 0 or not np.isfinite(s):
            probs = np.ones_like(probs) / probs.shape[0]
        else:
            probs = probs / s

        indices = np.random.choice(self.current_len, batch_size, p=probs)
        samples = [self.buffer[idx] for idx in indices]

        weights = (self.current_len * probs[indices]) ** (-beta)
        weights[~np.isfinite(weights)] = 1.0
        if weights.max() <= 0 or not np.isfinite(weights.max()):
            weights = np.ones_like(weights, dtype=np.float32)
        else:
            weights = weights / (weights.max() + 1e-12)

        batch = list(zip(*samples))
        states = np.array(batch[0])
        actions = np.array(batch[1])
        rewards = np.array(batch[2])
        next_states = np.array(batch[3])
        dones = np.array(batch[4]).astype(np.float32)

        return states, actions, rewards, next_states, dones, indices, weights.astype(np.float32)

    def update_priorities(self, indices, new_prios):
        new_prios = np.array(new_prios, dtype=np.float32)
        new_prios[~np.isfinite(new_prios)] = PRIO_EPS
        new_prios = np.clip(new_prios, PRIO_EPS, None)
        self.priorities[indices] = new_prios

        # Update raw TD-error (other criteria remain unchanged)
        for idx, new_td in zip(indices, new_prios):
            self.criteria_raw[idx, 0] = abs(new_td)

    def get_current_weights(self):
        return self.weights_cache.copy()

    def __len__(self):
        return self.current_len


# ========== Agent ==========
class MCDMDQNAgent:
    def __init__(self, state_dim, action_dim):
        self.device = device
        self.action_dim = action_dim
        self.batch_size = BATCH_SIZE
        self.gamma = GAMMA
        self.sync_steps = SYNC_TARGET_STEPS
        self.min_replay_size = MIN_REPLAY_SIZE

        self.online = QNet(state_dim, action_dim).to(device)
        self.target = QNet(state_dim, action_dim).to(device)
        self.target.load_state_dict(self.online.state_dict())
        self.opt = optim.Adam(self.online.parameters(), lr=LR)

        self.replay = MCDMPrioritizedReplayBuffer(BUFFER_SIZE, alpha=ALPHA)
        self.total_steps = 0
        self.eps_start, self.eps_final, self.eps_decay = EPS_START, EPS_FINAL, EPS_DECAY
        self.beta_start, self.beta_frames = BETA_START, BETA_FRAMES

    def epsilon(self):
        return self.eps_final + (self.eps_start - self.eps_final) * math.exp(-1.0 * self.total_steps / self.eps_decay)

    def beta_by_frame(self):
        return min(1.0, self.beta_start + self.total_steps * (1.0 - self.beta_start) / self.beta_frames)

    def act(self, state):
        if random.random() < self.epsilon():
            return random.randrange(self.action_dim)
        state_t = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            q = self.online(state_t)
        return int(q.argmax(dim=1).item())

    def act_continuous(self, state):
        discrete_action = self.act(state)
        return DISCRETE_ACTIONS_MAP[discrete_action]

    def push(self, s, a, r, s2, done):
        td_error = 1.0
        self.replay.push(s, a, r, s2, done, td_error=td_error)
        self.total_steps += 1
        if self.total_steps % self.sync_steps == 0:
            self.target.load_state_dict(self.online.state_dict())

    def learn(self):
        if len(self.replay) < self.min_replay_size:
            return None

        beta = self.beta_by_frame()
        batch = self.replay.sample(self.batch_size, beta=beta)
        if batch is None:
            return None
        states, actions, rewards, next_states, dones, indices, weights = batch

        s = torch.tensor(states, dtype=torch.float32, device=self.device)
        a = torch.tensor(actions, dtype=torch.int64, device=self.device).unsqueeze(1)
        r = torch.tensor(rewards, dtype=torch.float32, device=self.device).unsqueeze(1)
        s2 = torch.tensor(next_states, dtype=torch.float32, device=self.device)
        d = torch.tensor(dones, dtype=torch.float32, device=self.device).unsqueeze(1)
        w = torch.tensor(weights, dtype=torch.float32, device=self.device).unsqueeze(1)

        q_values = self.online(s).gather(1, a)
        next_actions = self.online(s2).argmax(dim=1, keepdim=True)
        next_q_target = self.target(s2).gather(1, next_actions)
        expected_q = r + (1.0 - d) * self.gamma * next_q_target

        td = q_values - expected_q.detach()
        loss = (w * td.pow(2)).mean()

        self.opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.online.parameters(), 10)
        self.opt.step()

        new_prios = td.detach().abs().mean(dim=1).cpu().numpy()
        new_prios[~np.isfinite(new_prios)] = PRIO_EPS
        new_prios = np.clip(new_prios, PRIO_EPS, None)
        self.replay.update_priorities(indices, new_prios)

        return loss.item()


# ========== Train single run ==========
def train_single_run(run_id):
    env = gym.make(ENV_NAME)
    state_dim = env.observation_space.shape[0]
    action_dim = NUM_DISCRETE_ACTIONS

    agent = MCDMDQNAgent(state_dim, action_dim)
    episode_rewards = []
    episode_losses = []
    episode_steps = []

    for ep in range(1, NUM_EPISODES + 1):
        state, _ = env.reset()
        done = False
        total_r = 0.0
        steps = 0
        ep_loss = []

        while not done:
            discrete_action = agent.act(state)
            continuous_action = DISCRETE_ACTIONS_MAP[discrete_action]

            next_state, reward, terminated, truncated, _ = env.step(continuous_action)
            done = terminated or truncated

            agent.push(state, discrete_action, reward, next_state, done)
            loss = agent.learn()
            if loss is not None:
                ep_loss.append(loss)
            state = next_state
            total_r += reward
            steps += 1

        episode_rewards.append(total_r)
        episode_losses.append(np.mean(ep_loss) if ep_loss else 0)
        episode_steps.append(steps)

        if ep % 20 == 0:
            avg20 = np.mean(episode_rewards[-20:]) if len(episode_rewards) >= 20 else np.mean(episode_rewards)
            w = agent.replay.get_current_weights()
            print(f"  Run {run_id} | Ep {ep}/{NUM_EPISODES} | Reward: {total_r:.2f} | Avg20: {avg20:.2f} | Eps: {agent.epsilon():.3f}")
            print(f"    Weights: TD={w[0]:.3f}, Rew={w[1]:.3f}, Nov={w[2]:.3f}, Rec={w[3]:.3f}")

    env.close()
    return episode_rewards, episode_losses, episode_steps, agent.replay.weights_history


# ========== Statistical Analysis ==========
def perform_statistical_analysis(all_rewards):
    final_scores = all_rewards[:, -20:].mean(axis=1)

    descriptive_stats = {
        'Mean': np.mean(final_scores),
        'Std': np.std(final_scores, ddof=1),
        'SEM': stats.sem(final_scores),
        'Median': np.median(final_scores),
        'Min': np.min(final_scores),
        'Max': np.max(final_scores),
        'Q1': np.percentile(final_scores, 25),
        'Q3': np.percentile(final_scores, 75),
        'IQR': np.percentile(final_scores, 75) - np.percentile(final_scores, 25),
        'Skewness': stats.skew(final_scores),
        'Kurtosis': stats.kurtosis(final_scores)
    }

    mean_score = descriptive_stats['Mean']
    sem_score = descriptive_stats['SEM']
    ci_95 = sem_score * stats.t.ppf(0.975, NUM_RUNS - 1)
    confidence_interval = (mean_score - ci_95, mean_score + ci_95)

    shapiro_stat, shapiro_p = shapiro(final_scores)
    is_normal = shapiro_p > 0.05
    levene_stat, levene_p = levene(final_scores, np.zeros_like(final_scores))

    cohens_d = mean_score / descriptive_stats['Std']
    if abs(cohens_d) < 0.2:
        effect_size_interpretation = "Very Small"
    elif abs(cohens_d) < 0.5:
        effect_size_interpretation = "Small"
    elif abs(cohens_d) < 0.8:
        effect_size_interpretation = "Medium"
    else:
        effect_size_interpretation = "Large"

    t_stat, t_p = ttest_ind(final_scores, np.zeros_like(final_scores))
    u_stat, u_p = mannwhitneyu(final_scores, np.zeros_like(final_scores), alternative='two-sided')

    results = {
        'Descriptive Statistics': descriptive_stats,
        'Confidence Interval 95%': confidence_interval,
        'Normality Test (Shapiro-Wilk)': {'statistic': shapiro_stat, 'p-value': shapiro_p, 'normal': is_normal},
        'Homogeneity (Levene)': {'statistic': levene_stat, 'p-value': levene_p},
        'Effect Size (Cohen\'s d)': {'value': cohens_d, 'interpretation': effect_size_interpretation},
        'T-test (vs zero)': {'statistic': t_stat, 'p-value': t_p, 'significant': t_p < 0.05},
        'Mann-Whitney U (vs zero)': {'statistic': u_stat, 'p-value': u_p, 'significant': u_p < 0.05}
    }

    return results, final_scores


# ========== Main Function ==========
def run_experiment():
    print(f"\n{'='*70}")
    print(f"MCDM-DQN (VIKOR Adaptive - FINAL FIX) on HalfCheetah-v5")
    print(f"Runs: {NUM_RUNS}, Episodes: {NUM_EPISODES}")
    print(f"✅ Initial weights: 0.25 (uniform)")
    print(f"✅ Weight update starts after {MIN_SAMPLES_FOR_WEIGHT_UPDATE} experiences")
    print(f"✅ VIKOR with Spearman correlation + absolute value")
    print(f"✅ True Recency based on insertion_times")
    print(f"{'='*70}\n")

    save_dir = "MCDM_DQN_HALFCHEETAH_FINAL_FIX"
    if os.path.exists(save_dir):
        shutil.rmtree(save_dir)
    os.makedirs(save_dir)

    all_rewards = []
    all_losses = []
    all_steps = []
    all_weight_histories = []

    for run in range(1, NUM_RUNS + 1):
        print(f"--- Run {run}/{NUM_RUNS} ---")
        rewards, losses, steps, weight_history = train_single_run(run)
        all_rewards.append(rewards)
        all_losses.append(losses)
        all_steps.append(steps)
        all_weight_histories.append(weight_history)

        df_run = pd.DataFrame({
            'Episode': range(1, NUM_EPISODES + 1),
            'Reward': rewards,
            'Loss': losses,
            'Steps': steps
        })
        df_run.to_excel(os.path.join(save_dir, f"run_{run}_results.xlsx"), index=False)
        print(f"  ✅ Run {run} completed! Avg Reward: {np.mean(rewards):.2f} ± {np.std(rewards):.2f}")

    weight_data = []
    for run_idx, history in enumerate(all_weight_histories):
        for step, weights in history:
            weight_data.append({
                'Run': run_idx + 1,
                'Step': step,
                'W_TD': weights[0],
                'W_Reward': weights[1],
                'W_Novelty': weights[2],
                'W_Recency': weights[3]
            })
    if weight_data:
        weight_df = pd.DataFrame(weight_data)
        weight_df.to_excel(os.path.join(save_dir, "adaptive_weights_history.xlsx"), index=False)

    rewards_array = np.array(all_rewards)
    losses_array = np.array(all_losses)
    steps_array = np.array(all_steps)

    print("\n" + "="*70)
    print("📊 Performing Statistical Analysis...")
    print("="*70)

    stats_results, final_scores = perform_statistical_analysis(rewards_array)

    desc_stats_df = pd.DataFrame({
        'Metric': list(stats_results['Descriptive Statistics'].keys()),
        'Value': list(stats_results['Descriptive Statistics'].values())
    })
    desc_stats_df.to_excel(os.path.join(save_dir, "descriptive_statistics.xlsx"), index=False)

    ci_df = pd.DataFrame({
        'Metric': ['Lower Bound (95%)', 'Upper Bound (95%)'],
        'Value': stats_results['Confidence Interval 95%']
    })
    ci_df.to_excel(os.path.join(save_dir, "confidence_interval.xlsx"), index=False)

    hypothesis_df = pd.DataFrame({
        'Test': ['Shapiro-Wilk (Normality)', 'Levene (Homogeneity)', 'Cohen\'s d (Effect Size)',
                'T-test (vs zero)', 'Mann-Whitney U (vs zero)'],
        'Statistic': [stats_results['Normality Test (Shapiro-Wilk)']['statistic'],
                     stats_results['Homogeneity (Levene)']['statistic'],
                     stats_results["Effect Size (Cohen's d)"]['value'],
                     stats_results['T-test (vs zero)']['statistic'],
                     stats_results['Mann-Whitney U (vs zero)']['statistic']],
        'p-value': [stats_results['Normality Test (Shapiro-Wilk)']['p-value'],
                   stats_results['Homogeneity (Levene)']['p-value'],
                   None,
                   stats_results['T-test (vs zero)']['p-value'],
                   stats_results['Mann-Whitney U (vs zero)']['p-value']],
        'Interpretation': [f"{'Normal' if stats_results['Normality Test (Shapiro-Wilk)']['normal'] else 'Non-normal'}",
                          f"{'Homogeneous' if stats_results['Homogeneity (Levene)']['p-value'] > 0.05 else 'Heterogeneous'}",
                          stats_results["Effect Size (Cohen's d)"]['interpretation'],
                          f"{'Significant' if stats_results['T-test (vs zero)']['significant'] else 'Not significant'}",
                          f"{'Significant' if stats_results['Mann-Whitney U (vs zero)']['significant'] else 'Not significant'}"]
    })
    hypothesis_df.to_excel(os.path.join(save_dir, "hypothesis_tests.xlsx"), index=False)

    summary_data = []
    for run in range(NUM_RUNS):
        summary_data.append({
            'Run': run + 1,
            'Mean Reward': np.mean(all_rewards[run]),
            'Std Reward': np.std(all_rewards[run]),
            'Min Reward': np.min(all_rewards[run]),
            'Max Reward': np.max(all_rewards[run]),
            'Mean Steps': np.mean(all_steps[run]),
            'Final Score (last 20)': np.mean(all_rewards[run][-20:]),
            'Total Loss': np.mean(all_losses[run])
        })

    summary_data.append({
        'Run': 'Overall Mean',
        'Mean Reward': np.mean([d['Mean Reward'] for d in summary_data]),
        'Std Reward': np.mean([d['Std Reward'] for d in summary_data]),
        'Min Reward': np.mean([d['Min Reward'] for d in summary_data]),
        'Max Reward': np.mean([d['Max Reward'] for d in summary_data]),
        'Mean Steps': np.mean([d['Mean Steps'] for d in summary_data]),
        'Final Score (last 20)': np.mean(final_scores),
        'Total Loss': np.mean([d['Total Loss'] for d in summary_data])
    })

    summary_df = pd.DataFrame(summary_data)
    summary_df.to_excel(os.path.join(save_dir, "summary.xlsx"), index=False)

    # ===== Plots =====
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    episodes = range(1, NUM_EPISODES + 1)
    mean_rewards = np.mean(rewards_array, axis=0)
    std_rewards = np.std(rewards_array, axis=0)
    sem_rewards = std_rewards / np.sqrt(NUM_RUNS)
    ci_95 = sem_rewards * stats.t.ppf(0.975, NUM_RUNS - 1)

    ax1 = axes[0, 0]
    ax1.plot(episodes, mean_rewards, '-', color='#2E86AB', label='Mean Reward', linewidth=2.5)
    ax1.fill_between(episodes, mean_rewards - ci_95, mean_rewards + ci_95,
                     alpha=0.25, color='#2E86AB', label='95% CI')
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Total Reward')
    ax1.set_title(f'MCDM-DQN Learning Curve ({NUM_RUNS} runs)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    parts = ax2.violinplot(final_scores, showmeans=True, showmedians=True)
    for pc in parts['bodies']:
        pc.set_facecolor('#2E86AB')
        pc.set_alpha(0.5)
        pc.set_edgecolor('black')
    bp = ax2.boxplot(final_scores, positions=[1], widths=0.2, patch_artist=True, showfliers=False)
    bp['boxes'][0].set_facecolor('white')
    bp['boxes'][0].set_edgecolor('#2E86AB')
    bp['medians'][0].set_color('red')
    ax2.set_xticks([1])
    ax2.set_xticklabels(['MCDM-DQN'])
    ax2.set_ylabel('Final Score (last 20 episodes)')
    ax2.set_title('Distribution of Final Scores')
    ax2.grid(True, alpha=0.3)

    ax3 = axes[0, 2]
    mean_loss = np.mean(losses_array, axis=0)
    std_loss = np.std(losses_array, axis=0)
    sem_loss = std_loss / np.sqrt(NUM_RUNS)
    ci_loss = sem_loss * stats.t.ppf(0.975, NUM_RUNS - 1)
    ax3.plot(episodes, mean_loss, '-', color='#D95B43', label='Mean Loss', linewidth=2.5)
    ax3.fill_between(episodes, mean_loss - ci_loss, mean_loss + ci_loss,
                     alpha=0.25, color='#D95B43', label='95% CI')
    ax3.set_xlabel('Episode')
    ax3.set_ylabel('Loss')
    ax3.set_title('Loss Curve with 95% CI')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 0]
    metrics = ['Mean', '95% CI\nLower', '95% CI\nUpper', "Cohen's\nd"]
    values = [
        stats_results['Descriptive Statistics']['Mean'],
        stats_results['Confidence Interval 95%'][0],
        stats_results['Confidence Interval 95%'][1],
        stats_results["Effect Size (Cohen's d)"]['value']
    ]
    colors = ['#2E86AB', '#A2D5F2', '#A2D5F2', '#6B4C3B']
    bars = ax4.bar(metrics, values, color=colors, alpha=0.75, edgecolor='black', linewidth=1.2)
    ax4.set_ylabel('Value')
    ax4.set_title('Key Statistical Metrics')
    ax4.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, values):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05 * max(values),
                f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax5 = axes[1, 1]
    if weight_data:
        weight_df_grouped = weight_df.groupby('Step').mean().reset_index()
        ax5.plot(weight_df_grouped['Step'], weight_df_grouped['W_TD'], 'r-', label='W_TD (TD-error)', linewidth=2)
        ax5.plot(weight_df_grouped['Step'], weight_df_grouped['W_Reward'], 'b-', label='W_Reward', linewidth=2)
        ax5.plot(weight_df_grouped['Step'], weight_df_grouped['W_Novelty'], 'g-', label='W_Novelty', linewidth=2)
        ax5.plot(weight_df_grouped['Step'], weight_df_grouped['W_Recency'], 'orange', label='W_Recency', linewidth=2)
        ax5.axhline(y=0.25, color='gray', linestyle='--', alpha=0.5, label='Initial (0.25)')
        ax5.set_xlabel('Training Steps')
        ax5.set_ylabel('Weight Value')
        ax5.set_title('Adaptive VIKOR Weights Evolution')
        ax5.legend()
        ax5.grid(True, alpha=0.3)

    ax6 = axes[1, 2]
    ax6.scatter(range(1, NUM_RUNS + 1), final_scores, s=100, c='#2E86AB', alpha=0.7, edgecolors='darkblue')
    ax6.axhline(y=np.mean(final_scores), color='red', linestyle='--', label=f'Mean: {np.mean(final_scores):.2f}')
    ax6.set_xlabel('Run Number')
    ax6.set_ylabel('Final Score')
    ax6.set_title('Final Scores Across Runs')
    ax6.legend()
    ax6.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "convergence_plots.png"), dpi=300)
    plt.show()

    np.savetxt(os.path.join(save_dir, "all_rewards.csv"), rewards_array, delimiter=",")
    np.savetxt(os.path.join(save_dir, "all_losses.csv"), losses_array, delimiter=",")
    np.savetxt(os.path.join(save_dir, "all_steps.csv"), steps_array, delimiter=",")

    report_text = f"""
================================================================
MCDM-DQN (VIKOR Adaptive - FINAL FIX) on HalfCheetah-v5
================================================================

Configuration:
- Number of Runs: {NUM_RUNS}
- Episodes per Run: {NUM_EPISODES}
- Initial Weights: 0.25 (uniform)
- Weight Learning Rate (η): {ETA_WEIGHT}
- Min Samples for Weight Update: {MIN_SAMPLES_FOR_WEIGHT_UPDATE}
- Correlation: Spearman (absolute value)
- Recency: based on real insertion times

================================================================
FINAL SCORES (last 20 episodes)
================================================================
  Mean: {np.mean(final_scores):.4f}
  Std: {np.std(final_scores, ddof=1):.4f}
  Median: {np.median(final_scores):.4f}
  Min: {np.min(final_scores):.4f}
  Max: {np.max(final_scores):.4f}
  95% CI: [{stats_results['Confidence Interval 95%'][0]:.4f}, {stats_results['Confidence Interval 95%'][1]:.4f}]
  Cohen's d: {stats_results["Effect Size (Cohen's d)"]['value']:.4f} ({stats_results["Effect Size (Cohen's d)"]['interpretation']})
  T-test p-value: {stats_results['T-test (vs zero)']['p-value']:.6e}
"""
    if weight_data:
        avg_weights = weight_df[['W_TD', 'W_Reward', 'W_Novelty', 'W_Recency']].mean()
        report_text += f"""
================================================================
AVERAGE ADAPTIVE WEIGHTS
================================================================
  W_TD: {avg_weights['W_TD']:.4f}
  W_Reward: {avg_weights['W_Reward']:.4f}
  W_Novelty: {avg_weights['W_Novelty']:.4f}
  W_Recency: {avg_weights['W_Recency']:.4f}
"""
    with open(os.path.join(save_dir, "report.txt"), 'w', encoding='utf-8') as f:
        f.write(report_text)

    print("\n" + "="*70)
    print("✅ All results saved successfully!")
    print(f"📁 Folder: {save_dir}")
    print("="*70)
    print(report_text)

    zip_filename = "MCDM_DQN_HalfCheetah_FINAL_FIX.zip"
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files_list in os.walk(save_dir):
            for file in files_list:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, save_dir)
                zipf.write(file_path, arcname)

    print(f"\n📦 ZIP file created: {zip_filename}")

    try:
        files.download(zip_filename)
        print("✅ Automatic download started!")
    except Exception as e:
        print(f"⚠️ Automatic download not possible: {e}")

    return save_dir, zip_filename


# ========== Execution ==========
if __name__ == "__main__":
    save_dir, zip_filename = run_experiment()